# XGBoost + GLCM — 5-Fold Patient-Level CV (BraTS 2021)

- Locked Test không được đọc trong notebook này.
- Development = Train + Validation.
- 5 folds chia theo `patient_id` bằng GroupKFold.
- Class weight được tính riêng từ training portion của từng fold.
- Xuất ROC-AUC/PR-AUC Fold 1–5, Mean, Variance, SD và OOF predictions.


In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix)

RANDOM_STATE=42
N_SPLITS=5
DATASET_PATH="dataset/1200p/glcm/kernel5/"
TRAIN_PATH=os.path.join(DATASET_PATH,"dataset.train.csv")
VAL_PATH=os.path.join(DATASET_PATH,"dataset.validation.csv")
LOCKED_TEST_PATH=os.path.join(DATASET_PATH,"dataset.test.csv")  # DO NOT LOAD HERE
OUTPUT_DIR="output/1200p/glcm/kernel5/patient_5fold_cv"
os.makedirs(OUTPUT_DIR,exist_ok=True)


## 1. Load Development only


In [ ]:
train_df=pd.read_csv(TRAIN_PATH)
val_df=pd.read_csv(VAL_PATH)
development=pd.concat([train_df.assign(original_split="train"),
                       val_df.assign(original_split="validation")],ignore_index=True)
print("Train",train_df.shape,"Validation",val_df.shape,"Development",development.shape)
print("Locked Test: NOT LOADED")
print(development.columns.tolist())


## 2. Validate patient metadata and leakage


In [ ]:
PATIENT_CANDIDATES=["patient_id","patient","subject_id","case_id"]
patient_col=next((c for c in PATIENT_CANDIDATES if c in development.columns),None)
if patient_col is None:
    raise ValueError("CSV thiếu patient_id. Không được chia 5-fold ngẫu nhiên theo voxel. Hãy sửa bước tạo dataset trước.")
if "label" not in development.columns:
    raise ValueError("CSV thiếu cột label.")

coord_sets=[("x","y","z"),("i","j","k"),("voxel_x","voxel_y","voxel_z")]
coord_cols=next((list(cs) for cs in coord_sets if all(c in development.columns for c in cs)),None)

train_patients=set(train_df[patient_col].astype(str).unique())
val_patients=set(val_df[patient_col].astype(str).unique())
overlap=train_patients & val_patients
if overlap:
    raise ValueError(f"Patient leakage Train/Validation: {len(overlap)} patients")

print("patient_col:",patient_col)
print("coordinates:",coord_cols)
print("development patients:",development[patient_col].nunique())
print(development["label"].value_counts().sort_index())


## 3. Select GLCM features


In [ ]:
META={"label","original_split",patient_col,"patient_id","patient","subject_id","case_id",
      "x","y","z","i","j","k","voxel_x","voxel_y","voxel_z"}
numeric=[c for c in development.columns if c not in META and pd.api.types.is_numeric_dtype(development[c])]
glcm=[c for c in numeric if "glcm" in c.lower()]
feature_cols=glcm if glcm else numeric
if not feature_cols:
    raise ValueError("Không tìm thấy numeric/GLCM features.")
print("features:",len(feature_cols))
print(*feature_cols,sep="\n")


## 4. Locked XGBoost configuration
Thay các giá trị này bằng cấu hình đã được chọn trên Train/Validation nếu tuning đã hoàn tất. Đây là cấu hình của luận văn, không phải cấu hình của Saeed et al.


In [ ]:
XGB_CONFIG=dict(
    n_estimators=500,max_depth=6,learning_rate=0.05,
    min_child_weight=1,gamma=0.0,subsample=0.8,colsample_bytree=0.8,
    reg_alpha=0.0,reg_lambda=1.0,objective="binary:logistic",
    eval_metric="aucpr",tree_method="hist",random_state=RANDOM_STATE,n_jobs=-1)
print(XGB_CONFIG)


## 5. 5-fold CV grouped by patient


In [ ]:
X=development[feature_cols]
y=development["label"].astype(int)
groups=development[patient_col].astype(str)
gkf=GroupKFold(n_splits=N_SPLITS)
rows=[]; oof_parts=[]

for fold,(tr_idx,va_idx) in enumerate(gkf.split(X,y,groups),1):
    Xtr,Xva=X.iloc[tr_idx],X.iloc[va_idx]
    ytr,yva=y.iloc[tr_idx],y.iloc[va_idx]
    gtr,gva=groups.iloc[tr_idx],groups.iloc[va_idx]
    if set(gtr.unique()) & set(gva.unique()):
        raise RuntimeError(f"Fold {fold}: patient leakage")
    if ytr.nunique()!=2 or yva.nunique()!=2:
        raise RuntimeError(f"Fold {fold}: thiếu một lớp")

    cls=np.array([0,1])
    w=compute_class_weight(class_weight="balanced",classes=cls,y=ytr)
    cw=dict(zip(cls,w))
    sw=ytr.map(cw).to_numpy()

    model=XGBClassifier(**XGB_CONFIG)
    model.fit(Xtr,ytr,sample_weight=sw,verbose=False)
    prob=model.predict_proba(Xva)[:,1]
    pred=(prob>=0.5).astype(int)
    tn,fp,fn,tp=confusion_matrix(yva,pred,labels=[0,1]).ravel()

    rows.append(dict(fold=fold,train_patients=gtr.nunique(),validation_patients=gva.nunique(),
        train_voxels=len(tr_idx),validation_voxels=len(va_idx),
        train_non_tumor=int((ytr==0).sum()),train_tumor=int((ytr==1).sum()),
        validation_non_tumor=int((yva==0).sum()),validation_tumor=int((yva==1).sum()),
        class_weight_0=float(cw[0]),class_weight_1=float(cw[1]),
        roc_auc=roc_auc_score(yva,prob),pr_auc=average_precision_score(yva,prob),
        accuracy_t05=accuracy_score(yva,pred),
        precision_t05=precision_score(yva,pred,zero_division=0),
        recall_t05=recall_score(yva,pred,zero_division=0),
        f1_t05=f1_score(yva,pred,zero_division=0),
        mcc_t05=matthews_corrcoef(yva,pred),
        tn_t05=int(tn),fp_t05=int(fp),fn_t05=int(fn),tp_t05=int(tp)))

    cols=[patient_col,"label"]+(coord_cols or [])
    part=development.iloc[va_idx][cols].copy()
    part["fold"]=fold; part["prob_tumor"]=prob; part["pred_t05"]=pred
    oof_parts.append(part)
    print(f"Fold {fold}: patients={gva.nunique()}, ROC-AUC={rows[-1]['roc_auc']:.4f}, PR-AUC={rows[-1]['pr_auc']:.4f}")


## 6. Fold 1–5, Mean, Variance, SD


In [ ]:
fold_results=pd.DataFrame(rows)
auc=fold_results["roc_auc"].to_numpy()
prauc=fold_results["pr_auc"].to_numpy()
summary=pd.DataFrame([{
 "model":"XGBoost + GLCM","modality":"FLAIR","n_folds":5,
 "roc_auc_mean":auc.mean(),"roc_auc_variance":np.var(auc,ddof=0),
 "roc_auc_sd_population":np.std(auc,ddof=0),"roc_auc_sd_sample":np.std(auc,ddof=1),
 "pr_auc_mean":prauc.mean(),"pr_auc_variance":np.var(prauc,ddof=0),
 "pr_auc_sd_population":np.std(prauc,ddof=0),"pr_auc_sd_sample":np.std(prauc,ddof=1)}])
fold_results.to_csv(os.path.join(OUTPUT_DIR,"xgb_glcm_patient_5fold_metrics.csv"),index=False)
summary.to_csv(os.path.join(OUTPUT_DIR,"xgb_glcm_patient_5fold_summary.csv"),index=False)
display(fold_results[["fold","validation_patients","roc_auc","pr_auc"]])
display(summary)


## 7. Saeed-style reporting table


In [ ]:
saeed_format=pd.DataFrame([{
 "Study":"Current study — XGBoost + GLCM","MRI":"FLAIR",
 "Fold 1":auc[0],"Fold 2":auc[1],"Fold 3":auc[2],"Fold 4":auc[3],"Fold 5":auc[4],
 "Mean AUC":auc.mean(),"Variance AUC":np.var(auc,ddof=0)}])
saeed_format.to_csv(os.path.join(OUTPUT_DIR,"table_saeed_format_current_study.csv"),index=False)
display(saeed_format)


## 8. OOF predictions + per-patient metrics


In [ ]:
oof=pd.concat(oof_parts,ignore_index=True)
oof.to_csv(os.path.join(OUTPUT_DIR,"development_oof_predictions.csv"),index=False)

patient_rows=[]
for pid,g in oof.groupby(patient_col):
    yt=g["label"].astype(int).to_numpy()
    yp=g["prob_tumor"].to_numpy()
    patient_rows.append({
      patient_col:pid,"n_voxels":len(g),
      "n_non_tumor":int((yt==0).sum()),"n_tumor":int((yt==1).sum()),
      "roc_auc":roc_auc_score(yt,yp) if np.unique(yt).size==2 else np.nan,
      "pr_auc":average_precision_score(yt,yp) if np.unique(yt).size==2 else np.nan})
patient_metrics=pd.DataFrame(patient_rows)
patient_metrics.to_csv(os.path.join(OUTPUT_DIR,"development_oof_patient_metrics.csv"),index=False)
display(patient_metrics.head())


## 9. Patient-level Mean / SD / 95% CI


In [ ]:
def summarize(s):
    x=pd.Series(s).dropna().astype(float).to_numpy()
    n=len(x); mean=x.mean() if n else np.nan
    sd=x.std(ddof=1) if n>1 else np.nan
    se=sd/np.sqrt(n) if n>1 else np.nan
    return dict(n=n,mean=mean,sd=sd,
      ci95_low=mean-1.96*se if n>1 else np.nan,
      ci95_high=mean+1.96*se if n>1 else np.nan,
      median=np.median(x) if n else np.nan,
      q1=np.quantile(x,.25) if n else np.nan,
      q3=np.quantile(x,.75) if n else np.nan)

patient_summary=pd.DataFrame([
 {"metric":"ROC-AUC",**summarize(patient_metrics["roc_auc"])},
 {"metric":"PR-AUC",**summarize(patient_metrics["pr_auc"])}])
patient_summary.to_csv(os.path.join(OUTPUT_DIR,"development_oof_patient_summary.csv"),index=False)
display(patient_summary)


# Protocol tiếp theo
Sau CV: không điều chỉnh model dựa trên Locked Test. Final Test dùng model + threshold đã khóa. Muốn tính whole-tumor segmentation Dice, phải dense inference trên toàn bộ voxel hợp lệ và reconstruct bằng `patient_id` + tọa độ 3D; không dùng riêng representative sampled voxels để gọi là whole-tumor Dice.
